# 5 · Day 3 – Gold, Delta & Reconciliation
Input: Silver tables in `capstone_5iot.silver`. Output: dimensions, facts and the daily KPI mart in the new schema **`capstone_5iot.gold`**.

| Task | What this notebook does |
|---|---|
| 1 | Data model: grain, keys, joins |
| 2 | Dimensions: `dim_machine`, `dim_plant`, `dim_failure_code`, `dim_date` |
| 3 | Facts: `fact_sensor_activity`, `fact_maintenance_episodes`, `fact_status_transitions` |
| 4 | Daily KPI mart `mart_sensor_daily` – built **incrementally** with MERGE |
| 5 | Reconciliation Gold vs Silver (TC07, TC08) + replay test (TC03) |
| 6 | Delta history, time travel, query plan, skew |
| 7 | Final validation summary |

## 0. Setup

In [0]:
from pyspark.sql import functions as F, Window


#hardcode for pipeline day 4 task 2
# dbutils.widgets.text("catalog", "capstone_5iot")
# dbutils.widgets.text("silver_schema", "silver")
# dbutils.widgets.text("gold_schema", "gold")
# CATALOG = dbutils.widgets.get("catalog")
# SILVER  = f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"
# GOLD    = f"{CATALOG}.{dbutils.widgets.get('gold_schema')}"

#org
SILVER = "capstone_5iot.silver"
GOLD   = "capstone_5iot.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
spark.sql(f"USE {GOLD}")                 # table names without a schema now point to gold

results = []

def check(task, name, actual, expected):
    status = "PASS" if actual == expected else "FAIL"
    results.append((task, name, str(actual), str(expected), status))
    print(f"[{status}] {task} | {name}: actual={actual} expected={expected}")

def save(df, table):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)

def silver(table):
    return spark.table(f"{SILVER}.{table}")

def date_key(col):
    return F.date_format(col, "yyyyMMdd").cast("int")      # 2026-07-01 -> 20260701

## 1. Data model

| Table | Grain (1 row =) | Keys |
|---|---|---|
| dim_machine | one machine | machine_sk · machine_id |
| dim_plant | one plant | plant_sk · plant_id |
| dim_failure_code | one failure code | failure_code_sk · failure_code |
| dim_date | one calendar day | date_key (yyyyMMdd) |
| fact_sensor_activity | one valid sensor reading | reading_id |
| fact_maintenance_episodes | one failure event (+ its work order) | failure_event_id |
| fact_status_transitions | one work-order status change | work_order_id + sequence_no |
| mart_sensor_daily | one machine per day | machine_id + activity_date |

**Surrogate key policy:** `*_sk = xxhash64(business key)` – same value on every run, so facts never need re-keying.
Facts keep both the surrogate key and the business key (easy to trace back to Silver/Bronze).

**Joins:** fact → dim_machine on machine_sk · fact → dim_plant on plant_sk (the reading's plant = the machine's plant) ·
fact_maintenance_episodes → dim_failure_code on failure_code_sk · all facts → dim_date on date_key.

## 2. Dimensions

In [0]:
dim_machine = silver("silver_machine_master").select(
    F.xxhash64("machine_id").alias("machine_sk"), F.xxhash64("plant_id").alias("plant_sk"), "*")

dim_plant = silver("silver_plant_master").select(F.xxhash64("plant_id").alias("plant_sk"), "*")

dim_failure_code = silver("silver_failure_code_lookup").select(F.xxhash64("failure_code").alias("failure_code_sk"), "*")

# one row per day between the first and last reading
days = silver("silver_sensor_readings").agg(F.min(F.to_date("reading_ts")).alias("first_day"),
                                            F.max(F.to_date("reading_ts")).alias("last_day"))
dim_date = (days.select(F.explode(F.sequence("first_day", "last_day")).alias("date"))
                .select(date_key("date").alias("date_key"), "date",
                        F.year("date").alias("year"), F.month("date").alias("month"),
                        F.dayofmonth("date").alias("day"), F.date_format("date", "EEEE").alias("day_name")))

save(dim_machine,      "dim_machine")
save(dim_plant,        "dim_plant")
save(dim_failure_code, "dim_failure_code")
save(dim_date,         "dim_date")

## 3. Facts

In [0]:
fact_sensor_activity = silver("silver_sensor_readings").select(
    "reading_id",
    F.xxhash64("machine_id").alias("machine_sk"), F.xxhash64("plant_id").alias("plant_sk"),
    date_key("reading_ts").alias("date_key"),
    "machine_id", "plant_id", "reading_ts",
    "temperature_c", "vibration_mm_s", "pressure_bar", "rpm",
    "operating_mode", "sensor_quality", "is_abnormal", "is_before_commissioning", "is_inactive_machine")

fact_maintenance_episodes = silver("silver_maintenance_episodes").select(
    "failure_event_id",
    F.xxhash64("machine_id").alias("machine_sk"), F.xxhash64("plant_id").alias("plant_sk"),
    F.xxhash64("failure_code").alias("failure_code_sk"), date_key("failure_ts").alias("date_key"),
    "machine_id", "plant_id", "failure_code", "work_order_id", "failure_ts",
    "severity", "downtime_minutes", "maintenance_type", "work_order_status", "repair_hours",
    "readings_before_failure", "abnormal_before_failure")

w = Window.partitionBy("work_order_id").orderBy("sequence_no")
fact_status_transitions = (silver("silver_work_order_status_history")
    .join(silver("silver_work_orders").select("work_order_id", "machine_id"), "work_order_id")
    .select("work_order_id", "sequence_no",
            F.xxhash64("machine_id").alias("machine_sk"), date_key("effective_ts").alias("date_key"),
            "machine_id",
            F.lag("status").over(w).alias("from_status"), F.col("status").alias("to_status"),
            "effective_ts"))

save(fact_sensor_activity,      "fact_sensor_activity")
save(fact_maintenance_episodes, "fact_maintenance_episodes")
save(fact_status_transitions,   "fact_status_transitions")

## 4. Daily KPI mart – `mart_sensor_daily`

| KPI | Definition (per machine per day) |
|---|---|
| **Equipment availability** | (1,440 − downtime minutes) / 1,440 – assumes 24 h planned run time |
| **Abnormal reading rate** | abnormal readings / valid readings (abnormal = temp > 85 °C or vibration > 7.1 mm/s or pressure > 18 bar) |
| **Maintenance downtime** | SUM(downtime_minutes) and number of failures |

**Incremental load:** the mart is filled with MERGE on (machine_id, activity_date).
Load 1 = July, Load 2 = August onwards, then Load 2 is replayed – the replay must not change or double count anything.

In [0]:
def daily_kpis(start_date, end_date):
    """KPIs per machine per day for days between start_date and end_date."""
    readings = (spark.table("fact_sensor_activity")
        .groupBy("machine_id", F.to_date("reading_ts").alias("activity_date"))
        .agg(F.count("*").alias("reading_count"),
             F.sum(F.col("is_abnormal").cast("int")).alias("abnormal_count")))

    downtime = (spark.table("fact_maintenance_episodes")
        .groupBy("machine_id", F.to_date("failure_ts").alias("activity_date"))
        .agg(F.count("*").alias("failure_count"),
             F.sum("downtime_minutes").alias("downtime_minutes")))

    return (readings.join(downtime, ["machine_id", "activity_date"], "full")
        .filter(F.col("activity_date").between(start_date, end_date))
        .fillna(0, ["reading_count", "abnormal_count", "failure_count", "downtime_minutes"])
        .join(spark.table("dim_machine").select("machine_id", "machine_sk", "plant_id", "plant_sk"), "machine_id")
        .withColumn("date_key", date_key("activity_date"))
        .withColumn("abnormal_reading_rate",
                    F.when(F.col("reading_count") > 0, F.round(F.col("abnormal_count") / F.col("reading_count"), 4)))
        .withColumn("availability", F.round((1440 - F.col("downtime_minutes")) / 1440, 4))
        .withColumn("updated_ts", F.current_timestamp()))

def merge_daily(df):
    df.createOrReplaceTempView("daily_updates")
    spark.sql("""
        MERGE INTO mart_sensor_daily t
        USING daily_updates s
        ON t.machine_id = s.machine_id AND t.activity_date = s.activity_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def mart_totals():
    return spark.table("mart_sensor_daily").agg(
        F.count("*"), F.sum("reading_count"), F.sum("abnormal_count"), F.sum("downtime_minutes")).first()[:]

In [0]:
spark.sql("DROP TABLE IF EXISTS mart_sensor_daily")
daily_kpis("2000-01-01", "2000-01-01").write.saveAsTable("mart_sensor_daily")   # empty table, correct columns

merge_daily(daily_kpis("2000-01-01", "2026-07-31"))      # load 1: July
after_load1 = mart_totals()
merge_daily(daily_kpis("2026-08-01", "2099-12-31"))      # load 2: August onwards
after_load2 = mart_totals()
merge_daily(daily_kpis("2026-08-01", "2099-12-31"))      # replay load 2
after_replay = mart_totals()

print("rows, readings, abnormal, downtime")
print("after load 1:", after_load1)
print("after load 2:", after_load2)
print("after replay:", after_replay)
display(spark.table("mart_sensor_daily").orderBy("activity_date", "machine_id").limit(20))

## 5. Reconciliation & tests
Gold is compared with Silver and with independent queries.

In [0]:
fsa  = spark.table("fact_sensor_activity")
fme  = spark.table("fact_maintenance_episodes")
fst  = spark.table("fact_status_transitions")
mart = spark.table("mart_sensor_daily")

def dupes(df, keys):
    return df.count() - df.select(keys).distinct().count()

def orphans(fact, key, dim):
    return spark.table(fact).join(spark.table(dim), key, "left_anti").count()

# --- dimensions ---
check("T2", "dim_machine rows",      spark.table("dim_machine").count(), 150)
check("T2", "dim_plant rows",        spark.table("dim_plant").count(), 159)
check("T2", "dim_failure_code rows", spark.table("dim_failure_code").count(), 168)
check("T2", "machine_sk unique",     dupes(spark.table("dim_machine"), ["machine_sk"]), 0)
check("T2", "plant_sk unique",       dupes(spark.table("dim_plant"), ["plant_sk"]), 0)

# --- facts: record reconciliation with Silver ---
check("T3", "fact_sensor_activity = silver readings",       fsa.count(), silver("silver_sensor_readings").count())
check("T3", "fact_maintenance_episodes = silver failures",  fme.count(), silver("silver_failure_events").count())
check("T3", "fact_status_transitions = silver history",     fst.count(), silver("silver_work_order_status_history").count())

# --- grain: keys unique ---
check("T5", "grain fact_sensor_activity (reading_id)",               dupes(fsa, ["reading_id"]), 0)
check("T5", "grain fact_maintenance_episodes (failure_event_id)",    dupes(fme, ["failure_event_id"]), 0)
check("T5", "grain fact_status_transitions (work_order_id, seq)",    dupes(fst, ["work_order_id", "sequence_no"]), 0)
check("T5", "grain mart_sensor_daily (machine_id, activity_date)",   dupes(mart, ["machine_id", "activity_date"]), 0)

# --- joins: no unmatched dimension keys (TC07) ---
check("T5", "fact_sensor_activity -> dim_machine",            orphans("fact_sensor_activity", "machine_sk", "dim_machine"), 0)
check("T5", "fact_sensor_activity -> dim_plant",              orphans("fact_sensor_activity", "plant_sk", "dim_plant"), 0)
check("T5", "fact_sensor_activity -> dim_date",               orphans("fact_sensor_activity", "date_key", "dim_date"), 0)
check("T5", "fact_maintenance_episodes -> dim_machine",       orphans("fact_maintenance_episodes", "machine_sk", "dim_machine"), 0)
check("T5", "fact_maintenance_episodes -> dim_failure_code",  orphans("fact_maintenance_episodes", "failure_code_sk", "dim_failure_code"), 0)
check("T5", "fact_maintenance_episodes -> dim_date",          orphans("fact_maintenance_episodes", "date_key", "dim_date"), 0)
check("T5", "fact_status_transitions -> dim_machine",         orphans("fact_status_transitions", "machine_sk", "dim_machine"), 0)

In [0]:
# --- metric reconciliation: mart totals = fact totals (TC08) ---
m = mart.agg(F.sum("reading_count").alias("r"), F.sum("abnormal_count").alias("a"),
             F.sum("failure_count").alias("f"), F.sum("downtime_minutes").alias("d")).first()

check("T5", "mart readings = fact readings",          m["r"], fsa.count())
check("T5", "mart abnormal = fact abnormal",          m["a"], fsa.filter("is_abnormal").count())
check("T5", "mart failures = fact failures",          m["f"], fme.count())
check("T5", "mart downtime = silver downtime (min)",  m["d"], silver("silver_failure_events").agg(F.sum("downtime_minutes")).first()[0])
check("T5", "total downtime minutes",                 m["d"], 3349)

# --- KPI vs independent SQL on Silver (TC08) ---
indep = spark.sql(f"""
    SELECT machine_id, to_date(failure_ts) AS activity_date,
           ROUND((1440 - SUM(downtime_minutes)) / 1440, 4) AS availability
    FROM {SILVER}.silver_failure_events
    GROUP BY machine_id, to_date(failure_ts)
""")
check("T5", "availability = independent SQL (machine-days with failures)",
      indep.join(mart, ["machine_id", "activity_date", "availability"], "left_anti").count(), 0)

rate_mart  = round(m["a"] / m["r"], 4)
rate_indep = round(spark.sql(f"SELECT AVG(CAST(is_abnormal AS INT)) FROM {SILVER}.silver_sensor_readings").first()[0], 4)
check("T5", "overall abnormal rate = independent SQL", rate_mart, rate_indep)
check("T5", "availability between 0 and 1",        mart.filter("availability < 0 OR availability > 1").count(), 0)

# --- incremental load (TC03) ---
full = daily_kpis("2000-01-01", "2099-12-31").drop("updated_ts")
check("T4", "load 2 added rows",                       after_load2[0] > after_load1[0], True)
check("T4", "replay changed nothing (no double count)", after_replay, after_load2)
check("T4", "incremental mart = full rebuild",
      mart.drop("updated_ts").exceptAll(full).count() + full.exceptAll(mart.drop("updated_ts")).count(), 0)

## 6. Delta history, time travel, query plan and skew

In [0]:
%sql
DESCRIBE HISTORY mart_sensor_daily

In [0]:
# time travel: the version written by the first MERGE = state after load 1
history     = spark.sql("DESCRIBE HISTORY mart_sensor_daily")
first_merge = history.filter("operation = 'MERGE'").agg(F.min("version")).first()[0]
rows_then   = spark.sql(f"SELECT COUNT(*) FROM mart_sensor_daily VERSION AS OF {first_merge}").first()[0]

print(f"version {first_merge}: {rows_then} rows   |   current: {mart.count()} rows")
check("T6", "time travel to load 1 version = load 1 rows", rows_then, after_load1[0])

In [0]:
%sql
EXPLAIN FORMATTED
SELECT p.region, d.date, AVG(m.availability) AS avg_availability, SUM(m.downtime_minutes) AS downtime
FROM mart_sensor_daily m
JOIN dim_plant p ON m.plant_sk = p.plant_sk
JOIN dim_date  d ON m.date_key = d.date_key
GROUP BY p.region, d.date

In [0]:
# skew: are readings spread evenly over machines?
per_machine = fsa.groupBy("machine_id").count().agg(F.min("count").alias("min"), F.max("count").alias("max"),
                                                    F.round(F.avg("count"), 1).alias("avg")).first()
print("readings per machine  min / max / avg:", per_machine["min"], per_machine["max"], per_machine["avg"])
check("T6", "no skew (largest machine < 2x average)", per_machine["max"] < 2 * per_machine["avg"], True)

## 7. Final validation summary

In [0]:
summary = spark.createDataFrame(results, "task string, check_name string, actual string, expected string, status string")
display(summary)

failed = summary.filter("status = 'FAIL'").count()
assert failed == 0, f"{failed} check(s) FAILED - see table above"
print(f"Day 3 complete - all {summary.count()} checks PASSED")

In [0]:
display(spark.sql(f"SHOW TABLES IN {GOLD}"))